<a href="https://colab.research.google.com/github/Georgegiri/cp5403-rag-assessment-helper/blob/main/notebooks/simple_rag_assistant_student_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate

In [33]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import textwrap

In [34]:
documents = [
    {
        "title": "Assessment Brief",
        "text": """
Assessment 2 is an individual LLM project. The project report is worth 30 percent and the oral presentation is worth 20 percent. Both are due in Week 10.

The written report should be 3000 to 4000 words. The oral presentation should be 15 minutes.

Students may choose any problem that interests them, but the problem should be suitable for an LLM application. Students should begin with an existing open-weights LLM and fine-tune the model and/or develop retrieval augmented generation capabilities.

Students are expected to find and organise appropriate data sources for their application. The report should include an introduction, target problem, choice of data sources, methodology, evaluation, discussion, conclusion and references.

Students are encouraged to develop the application within a GitHub repository to show good source code control.
"""
    },
    {
        "title": "GenAI Declaration",
        "text": """
Generative AI tools are not restricted for Assessment 2. Students may use GenAI tools to assist them in any way.

However, any use of generative AI must be appropriately acknowledged. Students must include a Declaration of AI Generated Material.

Students must still understand the technical details of their approach and must be able to answer questions about their project, data sources, methodology and evaluation.
"""
    },
    {
        "title": "Rubric Summary",
        "text": """
The assessment is marked on application and data selection, implementation code, evaluation code, report content, writing style, references, presentation content, speech, visuals, response to questions and reflection.

Higher marks require a relevant and sufficiently complex problem, appropriate data sources, effective LLM implementation, strong evaluation, accurate technical language, good academic writing and appropriate APA 7th edition references.

The final presentation must include a reflection explaining how the student responded to feedback, what changes were made, why those changes were made and how they improved the project.
"""
    }
]

print("Number of documents:", len(documents))

for doc in documents:
    print("-", doc["title"])

Number of documents: 3
- Assessment Brief
- GenAI Declaration
- Rubric Summary


In [35]:
chunks = []

for doc in documents:
    chunks.append({
        "title": doc["title"],
        "text": doc["text"].strip()
    })

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i}: {chunk['title']}")
    print(chunk["text"][:200], "...")

Number of chunks: 3

Chunk 0: Assessment Brief
Assessment 2 is an individual LLM project. The project report is worth 30 percent and the oral presentation is worth 20 percent. Both are due in Week 10.

The written report should be 3000 to 4000 wor ...

Chunk 1: GenAI Declaration
Generative AI tools are not restricted for Assessment 2. Students may use GenAI tools to assist them in any way.

However, any use of generative AI must be appropriately acknowledged. Students must in ...

Chunk 2: Rubric Summary
The assessment is marked on application and data selection, implementation code, evaluation code, report content, writing style, references, presentation content, speech, visuals, response to question ...


In [36]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = embedding_model.encode(chunk_texts)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("Embedding shape:", chunk_embeddings.shape)
print("Vector dimension:", dimension)
print("Number of chunks stored in FAISS:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (3, 384)
Vector dimension: 384
Number of chunks stored in FAISS: 3


In [37]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
llm_model = llm_model.to(device)

print("Model loaded on:", device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded on: cpu


In [46]:
def ask_rag(question, top_k=2, distance_threshold=1.5):
    # 1. Convert the user question into an embedding
    question_embedding = embedding_model.encode([question])
    question_embedding = np.array(question_embedding).astype("float32")

    # 2. Retrieve the most relevant chunks from FAISS
    distances, indices = index.search(question_embedding, top_k)

    # 3. Keep only chunks that are close enough
    retrieved_chunks = []
    retrieved_distances = []

    for idx, distance in zip(indices[0], distances[0]):
        if distance <= distance_threshold:
            retrieved_chunks.append(chunks[idx])
            retrieved_distances.append(distance)

    # 4. Display the question
    print("QUESTION:")
    print(question)

    print("\nRETRIEVED DOCUMENTS:")

    # 5. If no retrieved chunks are relevant enough, do not call the LLM
    if len(retrieved_chunks) == 0:
        print("No sufficiently relevant documents found.")
        print("\nANSWER:")
        print("The retrieved documents do not provide enough information.")
        return

    for chunk, distance in zip(retrieved_chunks, retrieved_distances):
        print(f"- {chunk['title']} | distance: {distance:.4f}")

    # 6. Combine retrieved chunks into one context block
    context = "\n\n".join(
        [f"Document: {chunk['title']}\n{chunk['text']}" for chunk in retrieved_chunks]
    )

    # 7. Build the RAG prompt
    prompt = f"""
You are an Assessment 2 student support assistant.

Answer the user's question using only the retrieved assessment documents below.

Rules:
1. Use only the retrieved documents as your source of truth.
2. If the answer is not explicitly found in the documents, say: "The retrieved documents do not provide enough information."
3. Do not infer, guess, or answer from general knowledge.
4. Do not create policies, penalties, deadlines, or requirements that are not stated in the documents.
5. Give a clear explanation in 3 to 5 sentences only when the answer is supported by the retrieved documents.

Retrieved documents:
{context}

User question:
{question}

Answer in a helpful way for a student:
"""

    # 8. Send the prompt to the language model
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=200,
        num_beams=4,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 9. Display the answer
    print("\nANSWER:")
    print(textwrap.fill(answer, width=100))

In [47]:
ask_rag("Can I use GenAI for Assessment 2?")

QUESTION:
Can I use GenAI for Assessment 2?

RETRIEVED DOCUMENTS:
- GenAI Declaration | distance: 0.6922
- Rubric Summary | distance: 1.1891

ANSWER:
Students may use GenAI tools to assist them in any way. However, any use of generative AI must be
appropriately acknowledged. Students must include a Declaration of AI Generated Material. Students
must still understand the technical details of their approach and must be able to answer questions
about their project, data sources, methodology and evaluation.


In [48]:
ask_rag("What should I include in my report?")

QUESTION:
What should I include in my report?

RETRIEVED DOCUMENTS:
- Rubric Summary | distance: 1.3386
- Assessment Brief | distance: 1.3474

ANSWER:
an introduction, target problem, choice of data sources, methodology, evaluation, discussion,
conclusion and references


In [49]:
ask_rag("How long is the oral presentation?")

QUESTION:
How long is the oral presentation?

RETRIEVED DOCUMENTS:
- Assessment Brief | distance: 1.3949

ANSWER:
15 minutes.


In [50]:
ask_rag("What should I include in my reflection?")

QUESTION:
What should I include in my reflection?

RETRIEVED DOCUMENTS:
- Rubric Summary | distance: 1.3095

ANSWER:
The final presentation must include a reflection explaining how the student responded to feedback,
what changes were made, why those changes were made and how they improved the project.


In [51]:
ask_rag("What is the penalty for late submission?")

QUESTION:
What is the penalty for late submission?

RETRIEVED DOCUMENTS:
No sufficiently relevant documents found.

ANSWER:
The retrieved documents do not provide enough information.
